# CSI Action/Idle Inference
Upload your `.keras` model and `csi_data.csv`, then run the cells below.

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

In [ ]:
# Change these paths if needed
MODEL_PATH = "/content/action_detection_model.keras"
CSV_PATH = "/content/csi_data.csv"

model = load_model(MODEL_PATH)
df = pd.read_csv(CSV_PATH)

print(df.head())
print("Rows:", len(df))

In [ ]:
# Compute amplitudes for 52 subcarriers
amp = []

for i in range(52):
    re = df[f"SC{i}_Re"].values
    im = df[f"SC{i}_Im"].values
    amplitude = np.sqrt(re**2 + im**2)
    amp.append(amplitude)

X = np.array(amp).T
print("Amplitude matrix:", X.shape)

In [ ]:
# Temporary normalization (replace with training scaler later)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

WINDOW_SIZE = 5

windows = []
for i in range(len(X)-WINDOW_SIZE+1):
    windows.append(X[i:i+WINDOW_SIZE])

windows = np.array(windows)
print("Window tensor:", windows.shape)

In [ ]:
pred = model.predict(windows, verbose=0)
pred_class = np.argmax(pred, axis=1)

idle = np.sum(pred_class == 0)
action = np.sum(pred_class == 1)

print(f"Idle windows   : {idle}")
print(f"Action windows : {action}")

if action > idle:
    print("\nFINAL PREDICTION: ACTION")
else:
    print("\nFINAL PREDICTION: IDLE")

In [ ]:
# Optional: visualize predictions
import matplotlib.pyplot as plt

plt.figure(figsize=(12,3))
plt.plot(pred_class)
plt.yticks([0,1],["Idle","Action"])
plt.xlabel("Window")
plt.ylabel("Prediction")
plt.grid(True)
plt.show()